In [1]:
import pyfredapi as pf

In [2]:
import pandas as pd
import pandas_datareader.data as web
import datetime

In [3]:
from fredapi import Fred
import os
from dotenv import load_dotenv
load_dotenv()

True

In [4]:
!pwd

/Users/jianbinchen/NonSync/GitHub/Research/PricingTailRisk_MDN/code


In [7]:
import pandas as pd
from fredapi import Fred

# 1. 初始化你的 API Key
fred_key = os.getenv('FRED_API_KEY')
fred = Fred(api_key=fred_key)

# ==========================================
# 场景 A：获取最新修正版的标准宏观数据
# ==========================================
# 获取美国核心 CPI (剔除食品和能源)
# 'CPILFESL' 是该指标在 FRED 上的官方 Ticker
cpi_latest = fred.get_series('CPILFESL', observation_start='2000-01-01')
print("--- 最新版核心 CPI ---")
print(cpi_latest.tail())

# ==========================================
# 场景 B：获取 ALFRED 实时快照数据 (发顶刊必备！)
# 解决 Look-ahead Bias 的核心代码
# ==========================================
# 获取历史上每一次发布和修正的记录
# 这个函数会返回一个 DataFrame，包含 'realtime_start' (该数值首次被市场看到的日期)
cpi_alfred = fred.get_series_all_releases('CPILFESL')

print("\n--- CPI 历史发布与修正记录 (ALFRED) ---")
print(cpi_alfred.head(10))

# 假设你的横截面日期是 2008年10月底
# 你可以通过过滤 realtime_start <= '2008-10-31'，来拿到当时市场真正能看到的最新数值！

--- 最新版核心 CPI ---
2025-11-01    331.043
2025-12-01    331.814
2026-01-01    332.793
2026-02-01    333.512
2026-03-01    334.165
dtype: float64

--- CPI 历史发布与修正记录 (ALFRED) ---
        realtime_start                 date value
0  1996-12-12 00:00:00  1957-01-01 00:00:00  28.5
1  1996-12-12 00:00:00  1957-02-01 00:00:00  28.6
2  1996-12-12 00:00:00  1957-03-01 00:00:00  28.7
3  1996-12-12 00:00:00  1957-04-01 00:00:00  28.8
4  1996-12-12 00:00:00  1957-05-01 00:00:00  28.8
5  1996-12-12 00:00:00  1957-06-01 00:00:00  28.9
6  1996-12-12 00:00:00  1957-07-01 00:00:00  29.0
7  1996-12-12 00:00:00  1957-08-01 00:00:00  29.0
8  1996-12-12 00:00:00  1957-09-01 00:00:00  29.1
9  1996-12-12 00:00:00  1957-10-01 00:00:00  29.2


In [8]:
cpi_alfred.tail(20)

,realtime_start,date,value
2235,2025-05-13 00:00:00,2025-04-01 00:00:00,326.43
2236,2026-02-13 00:00:00,2025-04-01 00:00:00,326.467
2237,2025-06-11 00:00:00,2025-05-01 00:00:00,326.854
2238,2026-02-13 00:00:00,2025-05-01 00:00:00,326.893
2239,2025-07-15 00:00:00,2025-06-01 00:00:00,327.6
2240,2026-02-13 00:00:00,2025-06-01 00:00:00,327.658
2241,2025-08-12 00:00:00,2025-07-01 00:00:00,328.656
2242,2026-02-13 00:00:00,2025-07-01 00:00:00,328.682
2243,2025-09-11 00:00:00,2025-08-01 00:00:00,329.793
2244,2026-02-13 00:00:00,2025-08-01 00:00:00,329.7


In [39]:
import pandas as pd
import numpy as np
import sqlite3
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import json
import copy
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)  # 显示所有列
pd.set_option('display.max_rows', 100)     # 显示前100行

# ==========================================
# 0. 自动设备检测
# ==========================================
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

# ==========================================
# 1. 外置数据预处理模块 (ETL & Feature Engineering)
# ==========================================
print(">>> [1/3] Loading and Preprocessing Data (Externalized)...")

micro_df = pd.read_parquet('./data/micro_data.parquet')
macro_df = pd.read_parquet('./data/macro_data.parquet')
micro_cols=micro_df.columns.drop(['mthcaldt', 'permno','mthret_lead1'])
macro_cols=macro_df.columns.drop('date')

Using device: mps
>>> [1/3] Loading and Preprocessing Data (Externalized)...


In [40]:
# 1.1 日期对齐与合并
micro_df['mthcaldt'] = pd.to_datetime(micro_df['mthcaldt']) + pd.offsets.MonthEnd(0)
macro_df['date'] = pd.to_datetime(macro_df['date']) + pd.offsets.MonthEnd(0)

df = pd.merge(micro_df, macro_df, left_on='mthcaldt', right_on='date', how='inner')
df = df.sort_values(['permno', 'mthcaldt']).reset_index(drop=True)


In [41]:
# 1.2 目标收益率设定 (绝对禁止缩尾！)
df['target_ret_final'] = df['mthret_lead1']

In [42]:
for feature in micro_cols:
    if feature in df.columns:
        df[feature] = df[feature].fillna(df.groupby('mthcaldt')[feature].transform('mean'))

In [43]:
# 1.4 微观特征：截面秩标准化 (Cross-Sectional Rank Normalization) [-1, 1]
print(" -> Applying Rank Normalization to Micro Features...")
micro_tensor_cols = []
for col in micro_cols:
    norm_col = f'{col}_norm'
    # 使用 pct=True 转化为 0~1，再映射到 -1~1
    df[norm_col] = df.groupby('mthcaldt')[col].transform(
        lambda x: (x.rank(pct=True) * 2) - 1
    ).fillna(0)  # 如果全截面缺失，填0(中性)
    micro_tensor_cols.append(norm_col)

# 1.5 宏观特征：时间序列 Z-score 标准化
print(" -> Applying Z-score Normalization to Macro Features...")
macro_tensor_cols = []
for col in macro_cols:
    norm_col = f'{col}_norm'
    df[norm_col] = (df[col] - df[col].mean()) / (df[col].std() + 1e-8)
    df[norm_col] = df[norm_col].fillna(0)
    macro_tensor_cols.append(norm_col)

 -> Applying Rank Normalization to Micro Features...
 -> Applying Z-score Normalization to Macro Features...


In [ ]:
processed_df = df.dropna(subset=['target_ret_final'] + micro_tensor_cols + macro_tensor_cols).copy()
print(f"Data preprocessing complete! Final shape: {processed_df.shape}")
print(f"Micro Features: {len(micro_tensor_cols)}, Macro Features: {len(macro_tensor_cols)}")

In [44]:
df.shape

(1629266, 95)

In [45]:
#print the first 5 rows and last 5 rows of the merged dataframe
df.iloc[[*range(5), *range(-5, 0)]]

,permno,mthcaldt,mthret,mthprc,mthcap,mthvol,shrout,short_debt,intcov_ratio,debt_at,debt_ebitda,ocf_lct,cash_ratio,accrual,bm,pe_exi,evm,divyield,roe,gpm,at_turn,rd_sale,mthret_lead1,turnover_1m,mthcap_log,MOM12m,date,AAA,BAA,lty,ltr,tbl,d/p,d/y,e/p,d/e,b/m,ntis,svar,disag,skvw,tail,rdsp,avgcor,tms,dfy,dfr,infl,vrp,target_ret_final,mthret_norm,mthprc_norm,mthcap_norm,mthvol_norm,shrout_norm,short_debt_norm,intcov_ratio_norm,debt_at_norm,debt_ebitda_norm,ocf_lct_norm,cash_ratio_norm,accrual_norm,bm_norm,pe_exi_norm,evm_norm,divyield_norm,roe_norm,gpm_norm,at_turn_norm,rd_sale_norm,turnover_1m_norm,mthcap_log_norm,MOM12m_norm,AAA_norm,BAA_norm,lty_norm,ltr_norm,tbl_norm,d/p_norm,d/y_norm,e/p_norm,d/e_norm,b/m_norm,ntis_norm,svar_norm,disag_norm,skvw_norm,tail_norm,rdsp_norm,avgcor_norm,tms_norm,dfy_norm,dfr_norm,infl_norm,vrp_norm
0,10001,1990-01-31,-0.018519,9.9375,1.015613e+04,3.525500e+04,1022,0.107954,3.131479,0.426239,2.590528,0.339098,0.271814,-0.005309,0.917331,2.743056,4.853678,0.050633,0.151031,0.134464,1.223659,0.000000,-0.006289,14.542074,9.225833,0.021139,1990-01-31,0.0899,0.0994,0.0865,-0.034300,0.0764,0.033860,0.031530,0.064714,0.483384,0.390455,-0.012334,0.002892,2.929758,0.049831,0.465337,0.022034,0.307570,0.0101,0.0095,0.015200,0.001589,35.9054,-0.006289,0.313613,0.222784,-0.429954,-0.699582,-0.952496,-0.467341,0.054322,0.594018,0.432593,0.287442,-0.039806,0.367055,0.507367,-0.359578,-0.261491,0.889158,0.316033,-0.539477,-0.086870,-0.336266,-0.364416,-0.429954,0.000440,1.685544,1.739982,1.778303,-1.451736,2.085981,2.255160,1.842672,1.573331,-0.019885,1.313355,-0.940412,0.044865,-0.936610,0.513882,0.480454,-0.654373,0.188809,-0.777180,0.080364,0.986846,-0.194865,0.737510
1,10001,1990-02-28,-0.006289,9.8750,1.009225e+04,1.486200e+04,1022,0.088959,3.119863,0.413490,2.279116,0.508677,0.273871,-0.024148,0.849363,2.544529,4.626443,0.050000,0.162685,0.139398,1.301493,0.000000,0.012821,12.392405,9.219523,0.015823,1990-02-28,0.0922,0.1014,0.0876,-0.002500,0.0774,0.033838,0.034126,0.068281,0.495891,0.414971,-0.013897,0.001009,3.042521,0.058325,0.476048,0.018495,0.316860,0.0102,0.0092,0.001300,0.010309,32.9177,0.012821,-0.235255,0.210607,-0.445863,-0.771127,-0.951144,-0.521127,0.087148,0.546215,0.346391,0.472271,-0.018926,0.226232,0.389965,-0.351673,-0.327025,0.888864,0.350792,-0.562060,0.006162,-0.417254,-0.523327,-0.445863,-0.000220,1.815662,1.855673,1.833550,-0.313290,2.131222,2.251432,2.270953,1.869971,0.013138,1.597576,-1.017996,-0.326998,-0.781470,0.700568,0.837300,-0.866763,0.264135,-0.769917,-0.003943,0.084123,2.483849,0.625327
2,10001,1990-03-31,0.012821,9.8750,1.014163e+04,1.272700e+04,1027,0.088959,3.119863,0.413490,2.279116,0.508677,0.273871,-0.024148,0.849363,2.576336,4.626443,0.049383,0.162685,0.139398,1.301493,0.000000,0.000000,16.207400,9.224404,-0.059086,1990-03-31,0.0937,0.1021,0.0889,-0.004400,0.0790,0.033294,0.034102,0.066498,0.508851,0.409173,-0.011729,0.001032,3.071526,0.009316,0.488723,0.023228,0.335480,0.0099,0.0084,0.003300,0.004710,26.5978,0.000000,-0.001539,0.191775,-0.447548,-0.830658,-0.954036,-0.525401,0.086870,0.548713,0.356939,0.470420,-0.028370,0.228942,0.381570,-0.356059,-0.310314,0.890917,0.352540,-0.559270,-0.000660,-0.421597,-0.349901,-0.447548,0.001979,1.900522,1.896164,1.898843,-0.381311,2.203608,2.161550,2.266863,1.721667,0.047357,1.530356,-0.910418,-0.322326,-0.741565,-0.376572,1.259604,-0.582725,0.415120,-0.791704,-0.228764,0.214011,0.763806,0.388024
3,10001,1990-04-30,0.000000,9.8750,1.014163e+04,1.664500e+04,1027,0.088959,3.119863,0.413490,2.279116,0.508677,0.273871,-0.024148,0.849363,2.544529,4.626443,0.050000,0.162685,0.139398,1.301493,0.000000,-0.012658,27.151899,9.224404,0.070617,1990-04-30,0.0946,0.1030,0.0924,-0.020200,0.0777,0.034562,0.033632,0.063747,0.522289,0.471334,-0.010291,0.000967,2.916158,0.007114,0.486923,0.023264,0.275801,0.0147,0.0084,0.001100,0.005469,26.2753,-0.012658,0.280914,0.206897,-0.444322,-0.786075,-0.953657,-0.522513,0.086317,0.553262,0.355150,0.467165

In [46]:
# print the rows with na in df
df[df.isna().any(axis=1)]

,permno,mthcaldt,mthret,mthprc,mthcap,mthvol,shrout,short_debt,intcov_ratio,debt_at,debt_ebitda,ocf_lct,cash_ratio,accrual,bm,pe_exi,evm,divyield,roe,gpm,at_turn,rd_sale,mthret_lead1,turnover_1m,mthcap_log,MOM12m,date,AAA,BAA,lty,ltr,tbl,d/p,d/y,e/p,d/e,b/m,ntis,svar,disag,skvw,tail,rdsp,avgcor,tms,dfy,dfr,infl,vrp,target_ret_final,mthret_norm,mthprc_norm,mthcap_norm,mthvol_norm,shrout_norm,short_debt_norm,intcov_ratio_norm,debt_at_norm,debt_ebitda_norm,ocf_lct_norm,cash_ratio_norm,accrual_norm,bm_norm,pe_exi_norm,evm_norm,divyield_norm,roe_norm,gpm_norm,at_turn_norm,rd_sale_norm,turnover_1m_norm,mthcap_log_norm,MOM12m_norm,AAA_norm,BAA_norm,lty_norm,ltr_norm,tbl_norm,d/p_norm,d/y_norm,e/p_norm,d/e_norm,b/m_norm,ntis_norm,svar_norm,disag_norm,skvw_norm,tail_norm,rdsp_norm,avgcor_norm,tms_norm,dfy_norm,dfr_norm,infl_norm,vrp_norm
219,10001,2008-02-29,0.022010,9.49990,41277.07,42393.0,4345,0.321263,30.535422,0.214650,8.796704e-01,0.100274,1.847215,-0.078817,0.580255,7.752855,3.327880e+00,0.026246,-0.029280,-11.095167,1.076460,9.551171,NaN,199.171184,10.628062,0.066943,2008-02-29,0.0553,0.0682,0.0438,0.0018,0.0212,0.021127,0.020392,0.046607,0.434579,0.262754,-0.043178,0.003226,3.856882,-0.068127,0.421695,0.034210,0.352288,0.0226,0.0129,-0.0089,0.004971,26.2320,NaN,0.356691,-0.131355,-0.732619,-0.946963,-0.924430,0.293212,0.575708,0.176147,0.102775,-0.359439,0.471009,-0.286068,0.222314,-0.205826,-0.469909,0.071173,-0.382248,-0.910415,0.187139,0.933498,0.262160,-0.732619,0.505908,-0.271890,-0.064787,-0.366313,-0.159349,-0.411323,0.149007,0.005635,0.067561,-0.148753,-0.167069,-2.471231,0.110729,0.338938,-2.078667,-0.973515,0.076290,0.551402,0.130605,1.035852,-0.578307,0.843977,0.374289
350,10005,1991-06-30,0.600000,0.12500,1047.00,150172.0,8376,0.000000,9.147241,0.020989,-8.075221e-02,-0.580645,1.193548,-0.319806,2.699726,-2.349999,-1.229673e+00,0.035749,-0.328391,-0.574713,0.200115,0.000000,NaN,61.554998,6.953684,0.666667,1991-06-30,0.0901,0.0996,0.0860,-0.0063,0.0557,0.032727,0.031160,0.051099,0.609170,0.439967,0.006982,0.001180,2.987926,0.036255,0.496793,0.020589,0.285339,0.0303,0.0095,0.0045,0.002959,26.4061,NaN,0.982942,-0.966549,-0.950377,-0.220647,0.068675,-0.970758,0.568232,-0.744794,-0.678777,-0.826318,0.494462,-0.917590,0.922907,-0.533895,-0.494019,0.084847,-0.716438,-0.945946,-0.904741,-0.436863,0.402525,-0.950377,0.801949,1.696858,1.751551,1.753190,-0.449331,1.149492,2.067767,1.781616,0.441096,0.312243,1.887348,0.018226,-0.293175,-0.856582,0.215497,1.528442,-0.741130,0.008550,0.689801,0.080364,0.291944,0.225959,0.380826
359,10007,1990-09-30,-0.464286,0.46875,1937.81,272982.0,4134,0.351314,11.222497,0.279283,-2.048514e+13,0.113474,1.360187,-0.052872,0.830635,8.020844,-3.086500e+13,0.040251,2.675467,-0.289013,1.332389,0.267428,NaN,50.600632,7.569314,-0.872146,1990-09-30,0.0956,0.1064,0.0914,0.0117,0.0736,0.038667,0.036688,0.066902,0.545737,0.488074,0.001439,0.001985,2.714018,0.039381,0.480156,0.030798,0.358001,0.0178,0.0108,-0.0026,0.009202,57.6982,NaN,-0.972179,-0.796202,-0.844116,0.152131,-0.414882,0.186796,0.609406,0.077501,-0.939501,-0.198719,0.606756,-0.060278,0.256127,0.029808,-0.940605,0.065577,0.890704,-0.862663,0.138883,0.880106,0.384632,-0.844116,-0.992051,2.008011,2.144898,2.024406,0.195073,1.959306,3.050240,2.693418,1.755293,0.144752,2.445046,-0.256868,-0.134259,-1.233428,0.284200,0.974171,-0.128475,0.597729,-0.217984,0.445698,-0.169159,2.143869,1.555797
426,10010,1995-07-31,0.091667,8.18750,84814.31,1246059.0,10359,0.342755,23.738143,0.230547,1.370410e+00,-0.027780,1.337123,-0.041652,0.641387,13.625808,7.239832e+00,0.027552,0.536699,-1.105506,1.343020,0.287306,NaN,125.956157,11.348220,0.463417,1995-07-31,0.0741,0.0804,0.0691,-0.0168,0.0542,0.023898,0.024658,0.063203,0.388063,0.286430,0.008276,0.000773,2.724456,0.009098,0.455933,0.022067,0.158588,0.0149,0.0063,0.0067,0.001971,10.2589,NaN,0.351920,-0.181170,-0.024863,0.247898,0.082267,0.193053,0.612066,0.104022,0.152468,-0.381536,0.552651,-0.068556,0.223766,0.01

In [47]:
processed_df = df.dropna(subset=['target_ret_final'] + micro_tensor_cols + macro_tensor_cols).copy()
print(f"Data preprocessing complete! Final shape: {processed_df.shape}")
print(f"Micro Features: {len(micro_tensor_cols)}, Macro Features: {len(macro_tensor_cols)}")

Data preprocessing complete! Final shape: (1616304, 95)
Micro Features: 23, Macro Features: 22


In [49]:
macro_tensor_cols

['AAA_norm',
 'BAA_norm',
 'lty_norm',
 'ltr_norm',
 'tbl_norm',
 'd/p_norm',
 'd/y_norm',
 'e/p_norm',
 'd/e_norm',
 'b/m_norm',
 'ntis_norm',
 'svar_norm',
 'disag_norm',
 'skvw_norm',
 'tail_norm',
 'rdsp_norm',
 'avgcor_norm',
 'tms_norm',
 'dfy_norm',
 'dfr_norm',
 'infl_norm',
 'vrp_norm']

In [36]:
processed_df.iloc[[*range(5), *range(-5, 0)]]

,permno,mthcaldt,mthret,mthprc,mthcap,mthvol,shrout,short_debt,intcov_ratio,debt_at,debt_ebitda,ocf_lct,cash_ratio,accrual,bm,pe_exi,evm,divyield,roe,gpm,at_turn,rd_sale,mthret_lead1,turnover_1m,mthcap_log,MOM12m,date,AAA,BAA,lty,ltr,tbl,d/p,d/y,e/p,d/e,b/m,ntis,svar,disag,skvw,tail,rdsp,avgcor,tms,dfy,dfr,infl,vrp,target_ret_final,mthret_norm,mthprc_norm,mthcap_norm,mthvol_norm,shrout_norm,short_debt_norm,intcov_ratio_norm,debt_at_norm,debt_ebitda_norm,ocf_lct_norm,cash_ratio_norm,accrual_norm,bm_norm,pe_exi_norm,evm_norm,divyield_norm,roe_norm,gpm_norm,at_turn_norm,rd_sale_norm,mthret_lead1_norm,turnover_1m_norm,mthcap_log_norm,MOM12m_norm,AAA_norm,BAA_norm,lty_norm,ltr_norm,tbl_norm,d/p_norm,d/y_norm,e/p_norm,d/e_norm,b/m_norm,ntis_norm,svar_norm,disag_norm,skvw_norm,tail_norm,rdsp_norm,avgcor_norm,tms_norm,dfy_norm,dfr_norm,infl_norm,vrp_norm
0,10001,1990-01-31,-0.018519,9.9375,1.015613e+04,3.525500e+04,1022,0.107954,3.131479,0.426239,2.590528,0.339098,0.271814,-0.005309,0.917331,2.743056,4.853678,0.050633,0.151031,0.134464,1.223659,0.000000,-0.006289,14.542074,9.225833,0.021139,1990-01-31,0.0899,0.0994,0.0865,-0.034300,0.0764,0.033860,0.031530,0.064714,0.483384,0.390455,-0.012334,0.002892,2.929758,0.049831,0.465337,0.022034,0.307570,0.0101,0.0095,0.015200,0.001589,35.9054,-0.006289,0.313613,0.222784,-0.429954,-0.699582,-0.952496,-0.467341,0.054322,0.594018,0.432593,0.287442,-0.039806,0.367055,0.507367,-0.359578,-0.261491,0.889158,0.316033,-0.539477,-0.086870,-0.336266,-0.238839,-0.364416,-0.429954,0.000440,1.685544,1.739982,1.778303,-1.451736,2.085981,2.255160,1.842672,1.573331,-0.019885,1.313355,-0.940412,0.044865,-0.936610,0.513882,0.480454,-0.654373,0.188809,-0.777180,0.080364,0.986846,-0.194865,0.737510
1,10001,1990-02-28,-0.006289,9.8750,1.009225e+04,1.486200e+04,1022,0.088959,3.119863,0.413490,2.279116,0.508677,0.273871,-0.024148,0.849363,2.544529,4.626443,0.050000,0.162685,0.139398,1.301493,0.000000,0.012821,12.392405,9.219523,0.015823,1990-02-28,0.0922,0.1014,0.0876,-0.002500,0.0774,0.033838,0.034126,0.068281,0.495891,0.414971,-0.013897,0.001009,3.042521,0.058325,0.476048,0.018495,0.316860,0.0102,0.0092,0.001300,0.010309,32.9177,0.012821,-0.235255,0.210607,-0.445863,-0.771127,-0.951144,-0.521127,0.087148,0.546215,0.346391,0.472271,-0.018926,0.226232,0.389965,-0.351673,-0.327025,0.888864,0.350792,-0.562060,0.006162,-0.417254,-0.015405,-0.523327,-0.445863,-0.000220,1.815662,1.855673,1.833550,-0.313290,2.131222,2.251432,2.270953,1.869971,0.013138,1.597576,-1.017996,-0.326998,-0.781470,0.700568,0.837300,-0.866763,0.264135,-0.769917,-0.003943,0.084123,2.483849,0.625327
2,10001,1990-03-31,0.012821,9.8750,1.014163e+04,1.272700e+04,1027,0.088959,3.119863,0.413490,2.279116,0.508677,0.273871,-0.024148,0.849363,2.576336,4.626443,0.049383,0.162685,0.139398,1.301493,0.000000,0.000000,16.207400,9.224404,-0.059086,1990-03-31,0.0937,0.1021,0.0889,-0.004400,0.0790,0.033294,0.034102,0.066498,0.508851,0.409173,-0.011729,0.001032,3.071526,0.009316,0.488723,0.023228,0.335480,0.0099,0.0084,0.003300,0.004710,26.5978,0.000000,-0.001539,0.191775,-0.447548,-0.830658,-0.954036,-0.525401,0.086870,0.548713,0.356939,0.470420,-0.028370,0.228942,0.381570,-0.356059,-0.310314,0.890917,0.352540,-0.559270,-0.000660,-0.421597,0.287442,-0.349901,-0.447548,0.001979,1.900522,1.896164,1.898843,-0.381311,2.203608,2.161550,2.266863,1.721667,0.047357,1.530356,-0.910418,-0.322326,-0.741565,-0.376572,1.259604,-0.582725,0.415120,-0.791704,-0.228764,0.214011,0.763806,0.388024
3,10001,1990-04-30,0.000000,9.8750,1.014163e+04,1.664500e+04,1027,0.088959,3.119863,0.413490,2.279116,0.508677,0.273871,-0.024148,0.849363,2.544529,4.626443,0.050000,0.162685,0.139398,1.301493,0.000000,-0.012658,27.151899,9.224404,0.070617,1990-04-30,0.0946,0.1030,0.0924,-0.020200,0.0777,0.034562,0.033632,0.063747,0.522289,0.471334,-0.010291,0.000967,2.916158,0.007114,0.486923,0.023264,0.275801,0.0147,0.0084,0.001100,0.005469,26.2753,-0.012658,0.280914,0.206897,-0.444322,-0.786075,-0.95365

In [12]:
macro_df.describe()

,date,AAA,BAA,lty,ltr,tbl,d/p,d/y,e/p,d/e,b/m,ntis,svar,disag,skvw,tail,rdsp,avgcor,tms,dfy,dfr,infl,vrp
count,408,408.000000,408.000000,408.000000,408.000000,408.000000,408.000000,408.000000,408.000000,408.000000,408.000000,408.000000,408.000000,408.000000,408.000000,408.000000,408.000000,408.000000,408.000000,408.000000,408.000000,408.000000,408.000000
mean,2007-01-14 15:10:35.294117632,0.056280,0.065820,0.046825,0.005859,0.026063,0.020130,0.020256,0.045981,0.496574,0.282869,0.003461,0.002752,3.710196,0.023051,0.452614,0.032458,0.299468,0.020762,0.009539,0.000230,0.002193,14.909037
min,1990-01-31 00:00:00,0.021400,0.031600,0.006200,-0.112400,0.000100,0.010849,0.010771,0.007935,0.288169,0.120510,-0.055954,0.000150,2.255945,-0.228505,0.345338,0.014326,0.072546,-0.015700,0.005500,-0.097600,-0.019153,-403.402100
25%,1998-07-23 06:00:00,0.040175,0.051300,0.029000,-0.012450,0.001800,0.016588,0.016720,0.039148,0.348975,0.222634,-0.012194,0.000774,3.048176,-0.000187,0.432980,0.022776,0.209727,0.010175,0.007075,-0.006325,0.000308,7.244350
50%,2007-01-15 12:00:00,0.054700,0.064350,0.047300,0.005550,0.022100,0.019095,0.019237,0.045064,0.407846,0.282539,0.006728,0.001423,3.560206,0.028473,0.454284,0.028083,0.285935,0.019650,0.008900,0.000400,0.002069,12.670600
75%,2015-07-07 18:00:00,0.071500,0.080225,0.062300,0.024225,0.048525,0.021492,0.021786,0.054636,0.510326,0.333668,0.016408,0.002727,4.281533,0.051649,0.471647,0.036988,0.369081,0.033050,0.010725,0.006625,0.004141,22.496275
max,2023-12-31 00:00:00,0.095600,0.107400,0.092400,0.144300,0.079000,0.039204,0.039480,0.076877,3.973032,0.522452,0.045727,0.073153,6.401161,0.158765,0.535641,0.161120,0.707789,0.045500,0.033800,0.073700,0.013736,115.853100
std,NaN,0.017979,0.017781,0.020302,0.028742,0.022242,0.005689,0.005697,0.012122,0.411842,0.082302,0.020321,0.005558,0.753237,0.047097,0.029604,0.015813,0.126637,0.014092,0.003706,0.016614,0.003429,29.048872
